In [ ]:
import sys
sys.path.append("../../")

import os
import numpy as np
import torch

import grid
import controller as NAZA_dQP
import controller_utils as utils_dQP
import dQP_mp as dQP

torch.set_default_dtype(torch.double)
torch.set_printoptions(threshold=10000)
NUMPY_SEED = 0
TORCH_SEED = 0
np.random.seed(NUMPY_SEED)
torch.manual_seed(TORCH_SEED)

skipping pardiso


In [ ]:
line_data_loc = '../data/case118_line_data.pt'
bus_data_loc = '../data/case118_bus_data.pt'
gen_data_loc = '../data/case118_gen_data.pt'
ptdf_data_loc = '../data/case118_ptdf_data.pt'

bus_with_curt = torch.load(gen_data_loc, weights_only=True)[:,0].int()
num_curt = bus_with_curt.shape[0]
bus_with_batt = torch.tensor([10*i+2 for i in range(12)], dtype=torch.int)
num_batt = bus_with_batt.shape[0]
delta_t = 15

grid = grid.Grid(bus_with_curt, bus_with_batt, delta_t, line_data_loc, bus_data_loc, gen_data_loc, ptdf_data_loc)
num_agents = 3
batt_cost = 100
curt_change_cost = 0.01
curt_net_cost = 1
bus_slack_cost = 1e8
line_slack_cost = 1e2

num_agents = 3
nodes_1 = [i for i in range(0, 42)] + [112, 113, 114, 116]
nodes_2 = [i for i in range(42, 69)] 
nodes_3 = [i for i in range(69, 112)] + [115, 117]
initial_state = grid.state
# agents = utils.create_split_constraint_agents(grid, num_agents, partition, target_charge, batt_cost, curt_net_cost, curt_change_cost)

bus_idx_gap = 10
T = 20
H = 5
file_path = "../../experiment_data/tauxDeChargeMTJLMA2juillet2018.txt"

ps_vals = [2, 2.4, 2.45, 2.5, 3, 3.5, 4]
offset_vals = [30 * i for i in range(10)]
train_skews = dict()

for ps in ps_vals:
    for offset in offset_vals:
        train_skews[(ps, offset)] = utils_dQP.get_RTE_noise_values(file_path, grid, T, H, ps, bus_idx_gap, offset=offset)
train_skews["ps_vals"] = ps_vals
train_skews["offset_vals"] = offset_vals

In [3]:
torch.save(train_skews, "train_skews.pt")

In [4]:
num_train_traj = 10
num_torch_seeds = 5
torch_seeds = [i for i in range(num_torch_seeds)]
train_radius = 0.2
train_trajs = dict()

for ps in ps_vals:
    for offset in offset_vals:
        train_skew = train_skews[(ps,offset)]
        for torch_seed in torch_seeds:
            torch.manual_seed(torch_seed)
            all_train_traj = torch.zeros(num_train_traj, T+H, grid.num_buses)
            for i in range(num_train_traj):
                dist_error = torch.rand_like(train_skew) * train_radius * 2 + (1-train_radius)
                all_train_traj[i] = dist_error * train_skew
            train_trajs[(ps, offset, torch_seed)] = all_train_traj
        train_trajs[(ps, offset)] = train_skew
train_trajs["num_train_traj"] = num_train_traj
train_trajs["num_torch_seeds"] = num_torch_seeds
train_trajs["train_radius"] = train_radius
            

In [5]:
torch.save(train_trajs, f"train_trajs_rad{train_radius}.pt")

In [22]:
num_test_traj = 10
test_trajs_no_mismatch = dict()
test_radius = 0.2
for ps in ps_vals:
    for offset in offset_vals:
        train_skew = train_skews[(ps,offset)]
        all_test_traj = torch.zeros(num_test_traj, T+H, grid.num_buses)
        for i in range(num_test_traj):
            dist_error = torch.rand_like(train_skew) * test_radius * 2 + (1-test_radius)
            all_test_traj[i] = dist_error * train_skew
        test_trajs_no_mismatch[(ps, offset)] = all_test_traj
test_trajs_no_mismatch["num_test_traj"] = num_test_traj
test_trajs_no_mismatch["test_radius"] = test_radius

In [23]:
torch.save(test_trajs_no_mismatch, f"test_trajs_no_mismatch_rad{test_radius}.pt")

In [24]:
num_test_traj = 10
test_radius = 0.2
test_skew_mags = [10, 20, 30, 40]
torch_seeds = [i for i in range(num_torch_seeds)]
folder = f"test_trajs_with_mismatch_rad{test_radius}"
os.makedirs(folder, exist_ok=True)
for ps in ps_vals:
    for offset in offset_vals:
        train_skew = train_skews[(ps,offset)]
        for test_skew_mag in test_skew_mags:
            test_trajs_with_mismatch = dict()
            for torch_seed in torch_seeds:
                torch.manual_seed(torch_seed)
                dist_error = torch.rand_like(train_skew) * (test_skew_mag / 100) * 2 + (1-test_skew_mag/100)
                test_skew = train_skew * dist_error
                all_test_traj = torch.zeros(num_test_traj, T+H, grid.num_buses)
                for i in range(num_test_traj):
                    dist_error = torch.rand_like(test_skew) * test_radius * 2 + (1-test_radius)
                    all_test_traj[i] = dist_error * test_skew
                test_trajs_with_mismatch[(ps,offset,test_skew_mag,torch_seed)] = all_test_traj
            test_trajs_with_mismatch["num_test_traj"] = num_test_traj
            test_trajs_with_mismatch["test_radius"] = test_radius
            test_trajs_with_mismatch["test_skew_mags"] = test_skew_mags
            test_trajs_with_mismatch["num_torch_seeds"] = num_torch_seeds
            torch.save(test_trajs_with_mismatch, f"{folder}/test_trajs_with_mismatch_ps{ps}_offset{offset}_tsm{test_skew_mag}.pt")
